<a href="https://colab.research.google.com/github/incrisvel/adaptaste/blob/main/semantic_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas


In [2]:
import pandas as pd
import json

# Abre arquivo de reviews
with open('reviews_preprocessed.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

preprocessed_reviews_df = pd.DataFrame(data)

# Guarda: o restante do trabalho assume o export com nota. Um arquivo antigo
# (sem rating) passaria em silêncio e geraria resultados errados.
required_fields = {"rating", "review_text"}
missing_fields = required_fields - set(preprocessed_reviews_df.columns)

if missing_fields:
    raise ValueError(
        f"Arquivo sem os campos obrigatórios {sorted(missing_fields)}: "
        "confira se é o reviews_preprocessed.json com rating."
    )

print(f"Reviews carregadas: {len(preprocessed_reviews_df)}")
print(f"Colunas: {list(preprocessed_reviews_df.columns)}")


Reviews carregadas: 2262
Colunas: ['review_id', 'review_date', 'place_id', 'review_text', 'rating', 'description_clean', 'description_tokenized', 'description_normalized', 'description_no_stopwords', 'description_stemmed', 'description_lemmatized', 'description_bigrams']


In [3]:
preprocessed_reviews_df

,review_id,review_date,place_id,review_text,rating,description_clean,description_tokenized,description_normalized,description_no_stopwords,description_stemmed,description_lemmatized,description_bigrams
0,Ci9DQUlRQUNvZENodHljRjlvT2xaVlRqSXpPRTFSYURKdl...,2026-07-30T17:09:02+00:00,0x6b0a883715bb4c6b:0,Excelente restaurante! A comida é muito saboro...,5.0,Excelente restaurante! A comida é muito saboro...,"[Excelente, restaurante, !, A, comida, é, muit...","[excelente, restaurante, a, comida, é, muito, ...","[excelente, restaurante, comida, muito, saboro...","[excel, restaurant, com, muit, sabor, bem, ser...","[excelente, restaurante, comer, muito, saboros...","[[excelente, restaurante], [restaurante, comer..."
1,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSl...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Fui pela primeira vez e gostei bastante! A com...,5.0,Fui pela primeira vez e gostei bastante! A com...,"[Fui, pela, primeira, vez, e, gostei, bastante...","[fui, pela, primeira, vez, e, gostei, bastante...","[primeira, vez, gostei, bastante, comida, muit...","[prim, vez, gost, bast, com, muit, sabor, peg,...","[primeiro, vez, gostar, bastante, comer, muito...","[[primeiro, vez], [vez, gostar], [gostar, bast..."
2,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONV...,2026-05-31T17:09:02+00:00,0x6b0a883715bb4c6b:0,Ambiente simples e gostoso familiar comida óti...,5.0,Ambiente simples e gostoso familiar comida óti...,"[Ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, e, gostoso, familiar, comi...","[ambiente, simples, gostoso, familiar, comida,...","[ambi, simpl, gost, famili, com, ótim, gost, m...","[ambiente, simples, gostoso, familiar, comer, ...","[[ambiente, simples], [simples, gostoso], [gos..."
3,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE,2022-08-30T17:10:53+00:00,0x6b0a883715bb4c6b:0,"Se é qualidade que procura, não tem outra igua...",5.0,"Se é qualidade que procura, não tem outra igua...","[Se, é, qualidade, que, procura, ,, não, tem, ...","[se, é, qualidade, que, procura, não, tem, out...","[qualidade, procura, não, outra, igual, agilid...","[qual, proc, não, outr, igual, agil, entreg, t...","[qualidade, procurar, não, outro, igual, agili...","[[qualidade, procurar], [procurar, não], [não,..."
4,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB,2022-08-30T17:10:52+00:00,0x6b0a883715bb4c6b:0,"Comida maravilhosa. Super caseira, bem tempera...",5.0,"Comida maravilhosa. Super caseira, bem tempera...","[Comida, maravilhosa, ., Super, caseira, ,, be...","[comida, maravilhosa, super, caseira, bem, tem...","[comida, maravilhosa, super, caseira, bem, tem...","[com, maravilh, sup, cas, bem, temper, tend, s...","[comida, maravilhoso, super, caseira, bem, tem...","[[comida, maravilhoso], [maravilhoso, super], ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
2257,ChdDSUhNMG9nS0VJQ0FnSUNiXzhYQnBnRRAB,2024-08-29T17:40:44+00:00,0x94df3b76767a0235:0,Perfeito ❤️🌹❤️ …,5.0,Perfeito …,"[Perfeito, …]",[perfeito],[perfeito],[perfeit],[perfeito],[perfeito]
2258,ChdDSUhNMG9nS0VJQ0FnSURiZ0pTWi13RRAB,2024-08-29T17:40:43+00:00,0x94df3b76767a0235:0,A melhor de todas! Recomendo,5.0,A melhor de todas! Recomendo,"[A, melhor, de, todas, !, Recomendo]","[a, melhor, de, todas, recomendo]","[melhor, todas, recomendo]","[melhor, tod, recom]","[bom, todo, recomendo]","[[bom, todo], [todo, recomendo]]"
2259,ChZDSUhNMG9nS0VJQ0FnSUNiXzYzX0ZnEAE,2024-08-29T17:40:43+00:00,0x94df3b76767a0235:0,Tudo ótimo e atendimento excelente!,5.0,Tudo ótimo e atendimento excelente!,"[Tudo, ótimo, e, atendimento, excelente, !]","[tudo, ótimo, e, atendimento, excelente]","[tudo, ótimo, atendimento, excelente]","[tud, ótim, atend, excel]","[tudo, ótimo, atendimento, excelente]","[[tudo, ótimo], [ótimo, atendimento], [atendim..."
2260,ChdDSUhNMG9nS0VJQ0FnSURiZ042MnZ3RRAB,2024-08-29T17:40:43+00:00,0x94df3b76767a0235:0,Marmitas deliciosas 😋🤩 …,5.0,Marmitas deliciosas …,"[Marmitas, deliciosas, …]","[marmitas, deliciosas]","[marmitas, deliciosas]","[marmit, delici]","[marmita, 

# Representação 1 — Word2Vec

In [4]:
%pip -q install gensim transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 51.8 MB/s eta 0:00:00


In [5]:
# Pipeline de preparação (híbrido) - MESMAS funções nos documentos e na consulta.
#
# - tokenização: word_tokenize (igual ao preprocessing.ipynb)
# - normalização: limpeza do professor (preserva hífen) + descarte de dígito
# - stopwords: lista PT menos negação/contraste, com 'muito' de volta às stopwords

import re

import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

for resource in ("punkt", "punkt_tab", "stopwords"):
    nltk.download(resource, quiet=True)

# Palavras preservadas (negacao, intensidade, restricao/contraste)
preserve_words = {
    # Negação
    "não",
    "nunca",
    "jamais",
    "nem",
    "sem",

    # Intensidade
    "pouco",
    "bastante",
    "demais",

    # Restrição/contraste
    "apenas",
    "só",
    "somente",
    "mas",
    "porém",
    "contudo"
}
# 'muito' fica de fora: token mais frequente do corpus (33% dos docs),
# alta diluicao e baixo poder discriminativo -> volta a ser stopword.

portuguese_stopwords = set(stopwords.words("portuguese")) - preserve_words


# 1. Tokenização
def tokenize(text):
    return word_tokenize(text, language="portuguese")


# 2. Normalização (minúsculas, sem pontuacao, preserva hifen, descarta digito)
def normalize(tokens):
    cleaned = [
        re.sub(r"[^\wÀ-ÿ-]", "", token.casefold()).strip("-")
        for token in tokens
    ]
    return [
        token
        for token in cleaned
        if token and not any(c.isdigit() for c in token)
    ]


# 3. Remoção de stopwords
def remove_stopwords(tokens):
    return [token for token in tokens if token not in portuguese_stopwords]


# Recalcular a representacao dos documentos com as MESMAS funções da consulta
preprocessed_reviews_df["description_no_stopwords"] = (
    preprocessed_reviews_df["description_tokenized"]
    .apply(normalize)
    .apply(remove_stopwords)
)


In [6]:
from gensim.models import Word2Vec

# Treinar Word2Vec no pequeno corpus da aula
w2v_model = Word2Vec(
    sentences=preprocessed_reviews_df["description_no_stopwords"].tolist(),
    vector_size=190,
    window=5,
    min_count=1,
    sg=1,          # Skip-gram
    seed=42,
    workers=1,     # ajuda na reprodutibilidade
    epochs=200
)

print("Tamanho do vocabulário:", len(w2v_model.wv))
print("Dimensões dos vetores:", w2v_model.vector_size)

# Algumas palavras disponíveis no vocabulário
list(w2v_model.wv.index_to_key)[:20]

Tamanho do vocabulário: 3861
Dimensões dos vetores: 190


['comida',
 'atendimento',
 'excelente',
 'bom',
 'ótimo',
 'lugar',
 'ambiente',
 'bem',
 'boa',
 'não',
 'recomendo',
 'melhor',
 'restaurante',
 'qualidade',
 'top',
 'super',
 'blumenau',
 'maravilhoso',
 'tudo',
 'sempre']

In [7]:
import numpy as np

def word2vec_document_vector(tokens):
    # Calcula a média dos vetores Word2Vec das palavras conhecidas.
    vectors = [
        w2v_model.wv[token]
        for token in tokens
        if token in w2v_model.wv
    ]

    if not vectors:
        return np.zeros(
            w2v_model.vector_size,
            dtype=np.float32
        )

    return np.mean(vectors, axis=0)


# Criar uma matriz: uma linha por descrição
embeddings_word2vec = np.vstack(
    preprocessed_reviews_df["description_no_stopwords"].apply(
        word2vec_document_vector
    )
)

# DataFrame apenas para inspecionar a representação
df_word2vec = pd.DataFrame(
    embeddings_word2vec,
    columns=[
        f"w2v_{i:03d}"
        for i in range(
            embeddings_word2vec.shape[1]
        )
    ]
)

df_word2vec.insert(
    0,
    "id",
    preprocessed_reviews_df["review_id"].values
)

print(
    "Formato da matriz Word2Vec:",
    embeddings_word2vec.shape
)

# Mostrar somente algumas dimensões
df_word2vec.iloc[:5, :8]

Formato da matriz Word2Vec: (2262, 190)


,id,w2v_000,w2v_001,w2v_002,w2v_003,w2v_004,w2v_005,w2v_006
0,Ci9DQUlRQUNvZENodHljRjlvT2xaVlRqSXpPRTFSYURKdl...,-0.076873,0.065425,0.158046,-0.056550,-0.338075,0.305784,-0.233818
1,Ci9DQUlRQUNvZENodHljRjlvT20xVFVVaHpaRVV4VVZWSl...,0.228516,-0.174449,0.071384,-0.093956,-0.324011,0.483285,-0.112071
2,Ci9DQUlRQUNvZENodHljRjlvT2psSmFEVkVNak5NTnpONV...,0.307861,-0.351133,0.147192,-0.042369,-0.227407,0.455932,-0.409505
3,ChZDSUhNMG9nS0VJQ0FnSUNHMWZEb1NBEAE,0.002814,-0.075773,0.205031,0.298125,-0.262578,0.262582,-0.266297
4,ChdDSUhNMG9nS0VJQ0FnSUNHc2NtMjRBRRAB,0.079707,-0.058226,-0.099547,-0.060351,-0.309030,0.613862,-0.164336


## Representação 2 — BERTimbau

In [8]:
import torch

from transformers import AutoTokenizer, AutoModel

BERT_MODEL = "neuralmind/bert-base-portuguese-cased"

# Detectar automaticamente GPU, quando disponível
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

# Carregar tokenizador e modelo
tokenizer_bert = AutoTokenizer.from_pretrained(
    BERT_MODEL,
    do_lower_case=False
)

bert_model = AutoModel.from_pretrained(
    BERT_MODEL
)

bert_model = bert_model.to(device)
bert_model.eval()

print("Modelo carregado:", BERT_MODEL)

Dispositivo: cuda


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Modelo carregado: neuralmind/bert-base-portuguese-cased


## Transformando uma sentença em um único vetor


In [9]:
def mean_pooling(
    last_hidden_state,
    attention_mask
):
    # Calcula a média dos vetores dos tokens,
    # ignorando posições utilizadas apenas como padding.
    mask = attention_mask.unsqueeze(-1).expand(
        last_hidden_state.size()
    ).float()

    total = torch.sum(
        last_hidden_state * mask,
        dim=1
    )

    count = torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

    return total / count


def generate_bert_embeddings(
    texts,
    batch_size=8
):
    # Gera um vetor por texto usando
    # BERTimbau + mean pooling.
    all_embeddings = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):
        batch = texts[
            start:start + batch_size
        ]

        inputs = tokenizer_bert(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        inputs = {
            key: value.to(device)
            for key, value
            in inputs.items()
        }

        with torch.no_grad():
            output = bert_model(**inputs)

        batch_embeddings = mean_pooling(
            output.last_hidden_state,
            inputs["attention_mask"]
        )

        all_embeddings.append(
            batch_embeddings.cpu().numpy()
        )

    return np.vstack(
        all_embeddings
    )


# Usamos a descrição ORIGINAL
texts = preprocessed_reviews_df["description_clean"].tolist()

embeddings_bert = generate_bert_embeddings(
    texts
)

print(
    "Formato da matriz BERT:",
    embeddings_bert.shape
)

Formato da matriz BERT: (2262, 768)


## Busca por similaridade

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def search_top_k(
    query_vector,
    document_matrix,
    k=3
):
    # Retorna os k documentos mais semelhantes
    # à consulta.
    query_vector = np.asarray(
        query_vector
    ).reshape(1, -1)

    similarities = cosine_similarity(
        query_vector,
        document_matrix
    )[0]

    indices = np.argsort(
        similarities
    )[::-1][:k]

    result = preprocessed_reviews_df.iloc[indices][["review_id", "place_id", "review_text"]].copy()

    result.insert(
        0,
        "position",
        range(1, len(result) + 1)
    )

    result["similarity"] = (
        similarities[indices]
    )

    return result

## Representando a consulta com Word2Vec

In [11]:
def word2vec_query_vector(
    text
):
    tokens = tokenize(text)
    tokens = normalize(tokens)
    tokens = remove_stopwords(tokens)

    known_tokens = [
        token
        for token in tokens
        if token in w2v_model.wv
    ]

    oov = [
        token
        for token in tokens
        if token not in w2v_model.wv
    ]

    if not known_tokens:
        return None, known_tokens, oov

    vector = np.mean(
        [
            w2v_model.wv[token]
            for token in known_tokens
        ],
        axis=0
    )

    return vector, known_tokens, oov

## Representando a consulta com BERTimbau

In [12]:
def bert_query_vector(
    text
):
    return generate_bert_embeddings(
        [text]
    )[0]

## TF-IDF

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Reconstruir o baseline TF-IDF da Aula 6
tfidf_vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    preprocessed_reviews_df["description_no_stopwords"]
)


def tfidf_search(
    query,
    k=3
):
    # Aplicar à consulta a mesma preparação
    # utilizada na Aula 6.
    tokens = tokenize(query)
    tokens = normalize(tokens)
    tokens = remove_stopwords(tokens)

    query_vector = (
        tfidf_vectorizer.transform(
            [tokens]
        )
    )

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    indices = np.argsort(
        similarities
    )[::-1][:k]

    result = preprocessed_reviews_df.iloc[indices][["review_id", "place_id", "review_text"]].copy()

    result.insert(
        0,
        "position",
        range(1, len(result) + 1)
    )

    result["similarity"] = (
        similarities[indices]
    )

    return result


## BOW

Contagens puras, sem IDF — é o TF-IDF com a ponderação desligada.

In [14]:
from sklearn.feature_extraction.text import CountVectorizer

# Baseline BOW: contagens puras, sem IDF
bow_vectorizer = CountVectorizer(
    analyzer=lambda tokens: tokens
)

bow_matrix = bow_vectorizer.fit_transform(
    preprocessed_reviews_df["description_no_stopwords"]
)


def bow_search(
    query,
    k=3
):
    # Aplicar à consulta a mesma preparação
    # utilizada nos documentos.
    tokens = tokenize(query)
    tokens = normalize(tokens)
    tokens = remove_stopwords(tokens)

    query_vector = (
        bow_vectorizer.transform(
            [tokens]
        )
    )

    similarities = cosine_similarity(
        query_vector,
        bow_matrix
    )[0]

    indices = np.argsort(
        similarities
    )[::-1][:k]

    result = preprocessed_reviews_df.iloc[indices][["review_id", "place_id", "review_text"]].copy()

    result.insert(
        0,
        "position",
        range(1, len(result) + 1)
    )

    result["similarity"] = (
        similarities[indices]
    )

    return result


## Executando uma busca

In [15]:
def compare_search(
    query,
    k=3
):
    print(f"\n{'='*50}\nConsulta: {query}\n{'='*50}")

    # 1. Representação Word2Vec
    w2v_vector, known_w2v, oov_w2v = word2vec_query_vector(query)

    print("\n--- Word2Vec ---")
    print(f"Tokens conhecidos: {known_w2v if known_w2v else 'Nenhum'}")
    print(f"Tokens fora do vocabulário (OOV): {oov_w2v if oov_w2v else 'Nenhum'}")

    if w2v_vector is not None:
        w2v_result = search_top_k(
            w2v_vector,
            embeddings_word2vec,
            k=k
        )
        print("\nResultados Word2Vec:")
        print(w2v_result.to_markdown(index=False))
    else:
        print("\nNão foi possível representar a consulta em Word2Vec (todos os tokens são OOV).")
        w2v_result = None

    # 2. Representação BERTimbau
    bert_vector = bert_query_vector(query)
    bert_result = search_top_k(
        bert_vector,
        embeddings_bert,
        k=k
    )
    print("\n--- BERTimbau ---")
    print("Resultados BERTimbau:")
    print(bert_result.to_markdown(index=False))

    # 3. Baseline TF-IDF
    tfidf_result = tfidf_search(
        query,
        k=k
    )
    print("\n--- TF-IDF ---")
    print("Resultados TF-IDF:")
    print(tfidf_result.to_markdown(index=False))

    # 4. Baseline BOW
    bow_result = bow_search(
        query,
        k=k
    )
    print("\n--- BOW ---")
    print("Resultados BOW:")
    print(bow_result.to_markdown(index=False))

    return {
        "word2vec": w2v_result,
        "bertimbau": bert_result,
        "tfidf": tfidf_result,
        "bow": bow_result
    }


query = input(
    "Digite sua consulta: "
).strip()

if not query:
    raise ValueError(
        "A consulta não pode estar vazia."
    )

result = compare_search(
    query,
    k=5
)

Digite sua consulta: Sem lactose

Consulta: Sem lactose

--- Word2Vec ---
Tokens conhecidos: ['sem', 'lactose']
Tokens fora do vocabulário (OOV): Nenhum

Resultados Word2Vec:
|   position | review_id                                                            | place_id             | review_text                                                                                                                            |   similarity |
|-----------:|:---------------------------------------------------------------------|:---------------------|:---------------------------------------------------------------------------------------------------------------------------------------|-------------:|
|          1 | Ci9DQUlRQUNvZENodHljRjlvT20xWFUxSk1OM0I0VWpVd1JUUkJRamxmWlVOR1QzYxAB | 0x94df1fc9d2166bf1:0 | Comida maravilhosa! Sem glúten! Sem lactose! Livre de contaminação!!!                                                                  |     0.897493 |
|          2 | Ci9DQUlRQUNvZENodHljRjlvT2p